# Figure 4 Supplementary — plot-data-only cache

This notebook is reorganized so that only the final data needed to redraw the UMAP parameter-sensitivity figure are cached.

The cache stores only:

- `n_neighbors_list`
- `min_dist_list`
- `all_spikes_label`
- all 25 UMAP embeddings
- the 25 silhouette scores
- the shared global plotting bounds

It does **not** store the original raw `.mat` inputs inside the cache.

**Workflow**
- **First run only:** load the raw features/labels once, compute the 25 embeddings and scores, then save only the final plot data.
- **Later runs:** load the cache and redraw the 5×5 panel figure directly.


In [ ]:
# ============================================================
# FIRST RUN ONLY
#
# This cell:
#   1) loads the original spectrogram features and manual labels
#   2) computes all 25 UMAP embeddings
#   3) computes the 25 silhouette scores
#   4) saves ONLY the final plot data
#
# After the cache exists, do NOT rerun this cell just to reproduce
# the figure.
# ============================================================

import os
import glob
import numpy as np
from scipy.io import loadmat
import umap
from sklearn.metrics import silhouette_score

pt_ch_mapping = {
    "EC130_200": [1, 13],
    "EC133_200": [282, 291, 292, 293],
    "EC135_200": [303],
    "EC137_200": [257, 258, 259, 269],
    "EC143_200": [339],
    "EC157_200": [321, 332, 333, 334],
    "EC162_200": [85, 86],
    "EC175_200": [301, 302],
    "EC183_200": [290],
    "EC186_200": [299, 300, 301],
    "EC187_200": [309, 310],
    "EC191_200": [2, 3, 4],
    "EC196_200": [300, 301, 303],
    "EC219_200": [399, 400, 402, 403],
    "EC220_200": [53, 54, 76, 77],
    "EC221_200": [376, 377],
    "EC222_200": [267, 268, 269, 270],
}

pathname = '/scratch/dazhang/MATLAB/'

all_specs_label = []
all_specs = np.empty((0, 60))
file_list = []

for patient, channal_list in pt_ch_mapping.items():
    print(patient, channal_list)
    path = pathname + patient + '/'
    ls_specs = np.empty((0, 60))

    for cur_channel in channal_list:
        cur_channel = str(cur_channel)
        for filename in glob.glob(os.path.join(path, '*.mat')):
            if ("ls_ripple" not in filename) or ("binsize_1_" in filename):
                continue
            if "Channel_" + str(cur_channel) + "_" in filename and ("-empty" not in filename):
                specs = loadmat(filename)['ls_ripple']
                file_list.append(filename)
                ls_specs = np.concatenate([ls_specs, specs], axis=0)

    all_specs_label = np.concatenate([all_specs_label, [patient] * len(ls_specs)])
    all_specs = np.concatenate([all_specs, ls_specs], axis=0)

print("all_specs:", all_specs.shape)

all_spikes_label = np.empty((0, 1))
for each_file in file_list:
    patient_name = each_file.split("/")[-2]
    actural_file = each_file.split("/")[-1]
    actural_file_list = actural_file.split("_")
    spike_file_name = (
        pathname
        + patient_name + "/"
        + actural_file_list[0]
        + "_spikes_"
        + "_".join(actural_file_list[2:6])
        + "_1_"
        + "_".join(actural_file_list[7:])
    )
    spikes = loadmat(spike_file_name)['ls_spikes']
    all_spikes_label = np.concatenate([all_spikes_label, spikes])

all_vectors = np.asarray(all_specs)
all_spikes_label = np.ravel(all_spikes_label)

print("all_vectors:", all_vectors.shape)
print("all_spikes_label:", all_spikes_label.shape)

n_neighbors_list = np.asarray([5, 15, 30, 100, 200], dtype=int)
min_dist_list = np.asarray([0.0, 0.1, 0.2, 0.3, 0.5], dtype=float)

n_rows = len(n_neighbors_list)
n_cols = len(min_dist_list)
n_points = all_vectors.shape[0]

embeddings = np.empty((n_rows, n_cols, n_points, 2), dtype=float)
scores = np.empty((n_rows, n_cols), dtype=float)

global_x_min = np.inf
global_x_max = -np.inf
global_y_min = np.inf
global_y_max = -np.inf

for i, n_nei in enumerate(n_neighbors_list):
    for j, min_d in enumerate(min_dist_list):
        print(f"Running UMAP: n_neighbors={n_nei}, min_dist={min_d}")

        embedding = umap.UMAP(
            n_components=2,
            min_dist=float(min_d),
            metric="euclidean",
            n_neighbors=int(n_nei),
            random_state=42,
            n_epochs=100,
            verbose=False
        ).fit_transform(all_vectors)

        actual_score = silhouette_score(embedding, all_spikes_label)

        embeddings[i, j] = embedding
        scores[i, j] = actual_score

        global_x_min = min(global_x_min, np.min(embedding[:, 0]))
        global_x_max = max(global_x_max, np.max(embedding[:, 0]))
        global_y_min = min(global_y_min, np.min(embedding[:, 1]))
        global_y_max = max(global_y_max, np.max(embedding[:, 1]))

x_pad = 0.05 * (global_x_max - global_x_min)
y_pad = 0.05 * (global_y_max - global_y_min)

x_min_plot = global_x_min - x_pad
x_max_plot = global_x_max + x_pad
y_min_plot = global_y_min - y_pad
y_max_plot = global_y_max + y_pad

print("Global x range:", x_min_plot, x_max_plot)
print("Global y range:", y_min_plot, y_max_plot)

CACHE_FILE = "Figure4_Suppl_plot_inputs.npz"
TMP_FILE = "Figure4_Suppl_plot_inputs.tmp.npz"

if os.path.exists(TMP_FILE):
    os.remove(TMP_FILE)

np.savez(
    TMP_FILE,
    n_neighbors_list=n_neighbors_list,
    min_dist_list=min_dist_list,
    all_spikes_label=all_spikes_label,
    embeddings=embeddings,
    scores=scores,
    x_min_plot=np.asarray(x_min_plot),
    x_max_plot=np.asarray(x_max_plot),
    y_min_plot=np.asarray(y_min_plot),
    y_max_plot=np.asarray(y_max_plot),
)

with np.load(TMP_FILE, allow_pickle=False) as check:
    _ = check["n_neighbors_list"]
    _ = check["min_dist_list"]
    _ = check["all_spikes_label"]
    _ = check["embeddings"]
    _ = check["scores"]
    _ = check["x_min_plot"]
    _ = check["x_max_plot"]
    _ = check["y_min_plot"]
    _ = check["y_max_plot"]

os.replace(TMP_FILE, CACHE_FILE)

print(f"Saved and verified: {CACHE_FILE}")
print("Future reproduction: skip this cell and use only the plot-only cells below.")


In [ ]:
# ============================================================
# PLOT ONLY LOADER — START HERE FOR ALL FUTURE REPRODUCTION
#
# No raw-data loading
# No UMAP fitting
# No silhouette recomputation
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

CACHE_FILE = "Figure4_Suppl_plot_inputs.npz"

with np.load(CACHE_FILE, allow_pickle=False) as cached:
    n_neighbors_list = cached["n_neighbors_list"]
    min_dist_list = cached["min_dist_list"]
    all_spikes_label = cached["all_spikes_label"]
    embeddings = cached["embeddings"]
    scores = cached["scores"]
    x_min_plot = float(cached["x_min_plot"])
    x_max_plot = float(cached["x_max_plot"])
    y_min_plot = float(cached["y_min_plot"])
    y_max_plot = float(cached["y_max_plot"])

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 12
})

c_list = []
for spike in all_spikes_label:
    if spike == 0:
        c_list.append([0, 0, 0])
    else:
        c_list.append([1, 0, 0])

print(f"Loaded: {CACHE_FILE}")
print("Plotting only — no upstream computation is being rerun.")


In [ ]:
# ============================================================
# PLOT ONLY — UMAP PARAMETER SENSITIVITY 5×5 GRID
# ============================================================

fig, axes = plt.subplots(
    nrows=len(n_neighbors_list),
    ncols=len(min_dist_list),
    figsize=(15, 15)
)

for i, n_nei in enumerate(n_neighbors_list):
    for j, min_d in enumerate(min_dist_list):
        ax = axes[i, j]
        embedding = embeddings[i, j]
        actual_score = scores[i, j]

        ax.scatter(
            embedding[:, 0],
            embedding[:, 1],
            c=c_list,
            s=1,
            linewidths=0,
            rasterized=True
        )

        ax.set_xlim(x_min_plot, x_max_plot)
        ax.set_ylim(y_min_plot, y_max_plot)
        ax.set_aspect("equal", adjustable="box")

        ax.set_xticks(np.linspace(x_min_plot, x_max_plot, 5))
        ax.set_yticks(np.linspace(y_min_plot, y_max_plot, 5))

        ax.tick_params(
            axis="both",
            which="both",
            length=0,
            labelbottom=False,
            labelleft=False
        )

        ax.grid(
            True,
            linestyle="-",
            linewidth=0.6,
            alpha=0.4
        )

        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

        if i == 0:
            ax.set_title(
                rf"$min\_dist = {float(min_d)}$",
                fontsize=12,
                pad=8
            )

        if j == 0:
            ax.text(
                -0.16,
                0.5,
                rf"$n\_neighbors = {int(n_nei)}$",
                transform=ax.transAxes,
                rotation=90,
                ha="center",
                va="center",
                fontsize=12
            )

        ax.text(
            0.03,
            0.97,
            f"Silhouette = {actual_score:.3f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            bbox=dict(
                facecolor="white",
                alpha=0.75,
                edgecolor="none",
                pad=1.2
            )
        )

fig.suptitle("Manual Labeled Spikes/Ripples", fontsize=16, y=0.985)

plt.subplots_adjust(
    left=0.08,
    right=0.995,
    bottom=0.02,
    top=0.94,
    wspace=0.05,
    hspace=0.08
)

# Optional:
# fig.savefig("UMAP_parameter_sensitivity_5x5_grid.png", dpi=600, bbox_inches="tight", pad_inches=0.03)
# fig.savefig("UMAP_parameter_sensitivity_5x5_grid.pdf", bbox_inches="tight", pad_inches=0.03)

plt.show()
plt.close(fig)


## Usage

**First run only**
1. Run `FIRST RUN ONLY` once.
2. It will create `Figure4_Suppl_plot_inputs.npz`.
3. Then run the plot-only cells if you want to draw the figure immediately.

**Every later run**
1. Skip the first-run cell entirely.
2. Start from `PLOT ONLY LOADER`.
3. Run `PLOT ONLY — UMAP PARAMETER SENSITIVITY 5×5 GRID`.

Only the final plot data are cached.
